In [ ]:
import os
import re
import shutil
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score
from pathlib import Path

**Input Folders:**

- Composite_outputs — contains subfolders named S2_[site]_[comp1]_[comp2] with STUMPF and XGBRF prediction CSVs derived from Sentinel-2 compositing methods

- Product_Composites — contains site folders (CC, COR, KW, UB) → pred_[site]_median or pred_[site]_median_out3std subfolders with STUMPF and XGBRF prediction CSVs derived from satellite data products

**Steps:**

- set up pattern matching requirements to find the files meeting the name structure

- **intersect_by_site** : For each site and method (STUMPF/XGBRF), it finds all prediction files from both folders and inner-joins them on lat/lon, keeping only pixels that appear across all compositions. This gives a single wide table per site/method where each composition has its own True_ and Predicted_ column pair.

- **report_mae_r2** — For each intersected table, computes MAE and R² for every composition and writes them to a .txt file.
**compile_results** — Parses all those .txt files into a single summary CSV.

- **report_single_file + compile_apples2oranges_results** — A separate, independent comparison that runs directly on the product files (median only) without intersecting against the composite data, saving its own MAE/R² txt files and summary CSV.

- extra functions for deleting previous .MAER2.txt files if you need to run this again

In [ ]:
# Pattern Matching

# Matches: STUMPF/XGBRF_val_predictions_S2_[site]_[comp1]_[comp2] (+ optional trailing data)
FILE_PATTERN_COMPOSITE = re.compile(
    r'^((STUMPF|XGBRF)_val_predictions_S2_[^_]+_[^_]+_[^_]+)',
    re.IGNORECASE
)

# Matches: STUMPF/XGBRF_val_predictions_pred_[site]_median (+ optional _out3std)
FILE_PATTERN_PRODUCT = re.compile(
    r'^((STUMPF|XGBRF)_val_predictions_pred_[^_]+_median)',
    re.IGNORECASE
)

SITE_TAGS = ['CC', 'COR', 'KW', 'UB']

SUBFOLDER_PATTERN = re.compile(r'^(pred_[^_]+_median(?:_out3std)?)$')


In [ ]:
# File collection ─────────────────────────────────────────────────────────

def get_csv_files_composite(folder_path: str) -> list:
    """Collect STUMPF/XGBRF csv files from a Composite_outputs subfolder."""
    return [
        os.path.join(folder_path, f) for f in os.listdir(folder_path)
        if FILE_PATTERN_COMPOSITE.match(f) and f.endswith('.csv')
    ]


def get_csv_files_product(folder_path: str) -> list:
    """Collect STUMPF/XGBRF csv files from a Product_Composites pred_* subfolder."""
    return [
        os.path.join(folder_path, f) for f in os.listdir(folder_path)
        if FILE_PATTERN_PRODUCT.match(f) and f.endswith('.csv')
    ]


def get_product_files(product_folder: str) -> list:
    """Walk Product_Composites (site → pred_* → csv) and collect all files."""
    files = []
    for site in SITE_TAGS:
        site_path = os.path.join(product_folder, site)
        if not os.path.isdir(site_path):
            continue
        for subfolder in os.listdir(site_path):
            subfolder_path = os.path.join(site_path, subfolder)
            if not os.path.isdir(subfolder_path):
                continue
            files.extend(get_csv_files_product(subfolder_path))
    return files


In [ ]:
# ─── Intersection ─────────────────────────────────────────────────────────────

def intersect_by_site(composite_folder: str, product_folder: str) -> dict:
    """
    For each site × method combination, inner-joins all prediction files
    from both folders on lat/lon. Saves a merged csv per site/method
    and returns a dict of dataframes.
    """
    methods = ['STUMPF', 'XGBRF']
    results = {}
    for site in SITE_TAGS:
        # Composite: S2_[site]_* folders directly under composite_folder
        composite_site_folders = [
            os.path.join(composite_folder, f)
            for f in os.listdir(composite_folder)
            if os.path.isdir(os.path.join(composite_folder, f))
            and len(f.split('_')) > 1
            and f.split('_')[1] == site
        ]
        # Product: pred_* folders under product_folder/site/
        site_path = os.path.join(product_folder, site)
        product_site_folders = [
            os.path.join(site_path, f)
            for f in os.listdir(site_path)
            if os.path.isdir(os.path.join(site_path, f))
        ] if os.path.isdir(site_path) else []
        for method in methods:
            dfs = []
            # Composite files
            for folder in composite_site_folders:
                folder_name = os.path.basename(folder)
                folder_name_clean = re.sub(r'_plusSCL_masked$', '', folder_name)
                tag_match = re.match(r'^(S2_[^_]+_[^_]+_[^_]+)', folder_name_clean)
                tag = tag_match.group(1) if tag_match else folder_name_clean

                # Try tagged filename first, fall back to untagged
                filepath_tagged = os.path.join(folder, f"{method}_val_predictions_{tag}.csv")
                filepath_untagged = os.path.join(folder, f"{method}_val_predictions.csv")

                if os.path.exists(filepath_tagged):
                    filepath = filepath_tagged
                elif os.path.exists(filepath_untagged):
                    filepath = filepath_untagged
                else:
                    print(f"Missing {method} predictions in {folder}")
                    continue

                df = pd.read_csv(filepath)
                composition = tag
                df = df[['lat', 'lon', 'True', 'Predicted']]
                df = df.rename(columns={
                    'True': f"True_{composition}",
                    'Predicted': f"Predicted_{composition}"
                })
                dfs.append(df)
            # Product files
            for folder in product_site_folders:
                subfolder_name = os.path.basename(folder)
                filename = f"{method}_val_predictions_{subfolder_name}.csv"
                filepath = os.path.join(folder, filename)
                if os.path.exists(filepath):
                    df = pd.read_csv(filepath)
                    composition = subfolder_name
                    df = df.rename(columns={'TRUE': 'True'})
                    df = df[['lat', 'lon', 'True', 'Predicted']]
                    df = df.rename(columns={
                        'True': f"True_{composition}",
                        'Predicted': f"Predicted_{composition}"
                    })
                    dfs.append(df)
                else:
                    print(f"Missing {method} file in {folder}")
            if not dfs:
                print(f"No files found for {site} - {method}")
                continue
            merged = dfs[0]
            for df in dfs[1:]:
                merged = merged.merge(df, on=['lat', 'lon'], how='inner')
            output_path = os.path.join(composite_folder, f"{site}_{method}_intersected.csv")
            merged.to_csv(output_path, index=False)
            print(f"Saved: {output_path}")
            results[f"{site}_{method}"] = merged
    return results

In [ ]:
#  Metrics 
def report_mae_r2(intersected: dict, parent_folder: str) -> None:
    """Writes a .txt file per site/method with MAE and R² for each composition."""
    for key, df in intersected.items():
        output_path = os.path.join(parent_folder, f"{key}_MAER2.txt")
        compositions = [c.replace('True_', '') for c in df.columns if c.startswith('True_')]

        with open(output_path, 'w') as f_out:
            f_out.write(f"Results for: {key}\n")
            f_out.write(f"Number of pixels: {len(df)}\n\n")

            for composition in compositions:
                true_col = f"True_{composition}"
                pred_col = f"Predicted_{composition}"

                if true_col not in df.columns or pred_col not in df.columns:
                    print(f"Skipping {composition}: missing True or Predicted column.")
                    continue

                mae = mean_absolute_error(df[true_col], df[pred_col])
                r2 = r2_score(df[true_col], df[pred_col])

                f_out.write(f"Composition: {composition}\n")
                f_out.write(f"  MAE: {mae:.4f}\n")
                f_out.write(f"  R2:  {r2:.4f}\n\n")

        print(f"Saved: {output_path}")


def compile_results(parent_folder: str) -> pd.DataFrame:
    """Parses all _MAER2.txt files into a single summary DataFrame."""
    results = []
    txt_files = [f for f in os.listdir(parent_folder) if f.endswith('MAER2.txt')]

    if not txt_files:
        print("No MAER2.txt files found.")
        return pd.DataFrame()

    for txt_file in txt_files:
        txt_path = os.path.join(parent_folder, txt_file)
        try:
            with open(txt_path, 'r') as f:
                content = f.read()

            parts = txt_file.replace('_MAER2.txt', '').split('_')
            site = parts[0]
            method = parts[1]

            n_pixels = int(
                [l for l in content.splitlines() if 'Number of pixels' in l][0].split(': ')[1]
            )

            blocks = content.split('Composition: ')[1:]
            for block in blocks:
                lines = block.strip().splitlines()
                composition = lines[0].strip()
                mae = float(lines[1].split('MAE: ')[1])
                r2 = float(lines[2].split('R2: ')[1])

                results.append({
                    'site': site,
                    'method': method,
                    'composition': composition,
                    'n_pixels': n_pixels,
                    'mae': mae,
                    'r2': r2
                })

        except Exception as e:
            print(f"Error reading {txt_file}: {e}")

    df = pd.DataFrame(results).sort_values(['site', 'method', 'composition']).reset_index(drop=True)
    return df

In [ ]:
# Pipelines
def run_pipeline(composite_folder: str, product_folder: str) -> pd.DataFrame:
    """Full end-to-end pipeline."""
    print("\nIntersecting by site and method...")
    intersected = intersect_by_site(composite_folder, product_folder)

    print("\nReporting MAE and R2...")
    report_mae_r2(intersected, composite_folder)

    print("\nCompiling results...")
    results_df = compile_results(composite_folder)
    results_df.to_csv(os.path.join(composite_folder, "results_summary_apples2apples.csv"), index=False)
    print(f"Saved: {os.path.join(composite_folder, 'results_summary_apples2apples.csv')}")

    return results_df

In [ ]:
results_df = run_pipeline(r"M:\SDB_Compare\Composite_outputs", r"M:\SDB_Compare\Product_Composites")
#print(results_df)

In [ ]:
# results_df.to_csv(r"M:\SDB_Compare\Composite_outputs\results_summary_apples2apples.csv", index=False)

# Get the Apples to Oranges (all available pixels) MAE data from the Product Composite folder

In [ ]:
# NEEDS EXACT PATH
REPORT_PATTERN = re.compile(
    r'^(STUMPF|XGBRF)_val_predictions_pred_[^_]+_median',
    re.IGNORECASE
)

def report_single_file(folder: str) -> None:
    matches = [
        f for f in os.listdir(folder)
        if REPORT_PATTERN.match(f) and f.endswith('.csv') and '_latlon' not in f
    ]

    if not matches:
        print(f"No matching median csv found in: {folder}")
        return

    for match in matches:
        filepath = os.path.join(folder, match)
        df = pd.read_csv(filepath)
        df = df.rename(columns={'TRUE': 'True'})
        if 'True' not in df.columns or 'Predicted' not in df.columns:
            print(f"Missing 'True' or 'Predicted' column in {filepath}")
            continue

        n_pixels = len(df)
        mae = mean_absolute_error(df['True'], df['Predicted'])
        r2 = r2_score(df['True'], df['Predicted'])
        base, _ = os.path.splitext(filepath)
        output_path = f"{base}_apples2oranges_MAER2.txt"

        with open(output_path, 'w') as f_out:
            f_out.write(f"File: {os.path.basename(filepath)}\n")
            f_out.write(f"Number of pixels: {n_pixels}\n")
            f_out.write(f"MAE: {mae:.4f}\n")
            f_out.write(f"R2: {r2:.4f}\n")
        print(f"Saved: {output_path}")


In [ ]:
SITE_TAGS = ['CC', 'COR', 'KW', 'UB']
product_folder = r"M:\SDB_Compare\Product_Composites"

for site in SITE_TAGS:
    site_path = os.path.join(product_folder, site)
    if not os.path.isdir(site_path):
        continue
    for subfolder in os.listdir(site_path):
        subfolder_path = os.path.join(site_path, subfolder)
        if os.path.isdir(subfolder_path):
            report_single_file(subfolder_path)

# Get the A2O Product Composite MAE data into a single .csv

In [ ]:
def compile_apples2oranges_results(product_folder: str) -> pd.DataFrame:
    results = []

    for site in SITE_TAGS:
        site_path = os.path.join(product_folder, site)
        if not os.path.isdir(site_path):
            continue
        for subfolder in os.listdir(site_path):
            subfolder_path = os.path.join(site_path, subfolder)
            if not os.path.isdir(subfolder_path):
                continue
            for f in os.listdir(subfolder_path):
                if not f.endswith('apples2oranges_MAER2.txt'):
                    continue
                txt_path = os.path.join(subfolder_path, f)
                try:
                    with open(txt_path, 'r') as f_in:
                        lines = f_in.read().splitlines()
                    filename  = lines[0].split('File: ')[1]
                    n_pixels  = int(lines[1].split('Number of pixels: ')[1])
                    mae       = float(lines[2].split('MAE: ')[1])
                    r2        = float(lines[3].split('R2: ')[1])

                    results.append({
                        'site':       site,
                        'subfolder':  subfolder,
                        'file':       filename,
                        'n_pixels':   n_pixels,
                        'mae':        mae,
                        'r2':         r2
                    })
                except Exception as e:
                    print(f"Error reading {txt_path}: {e}")

    df = pd.DataFrame(results).sort_values(['site', 'subfolder']).reset_index(drop=True)
    output_path = os.path.join(product_folder, "apples2oranges_summary.csv")
    df.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")
    return df

In [ ]:
df = compile_apples2oranges_results(r"M:\SDB_Compare\Product_Composites")

# Helpful functions where to need to re-run this but want to remove old files

In [ ]:
def remove_maer2_files(parent_folder: str) -> None:
    """Deletes all files ending in MAER2.txt in the given folder and all subfolders."""
    removed = 0
    for root, dirs, files in os.walk(parent_folder):
        for f in files:
            if f.endswith('MAER2.txt'):
                filepath = os.path.join(root, f)
                os.remove(filepath)
                print(f"Removed: {filepath}")
                removed += 1
    if removed == 0:
        print("No MAER2.txt files found.")
    else:
        print(f"\nRemoved {removed} file(s).")

In [ ]:
remove_maer2_files(r'M:\SDB_Compare\Product_Composites')
#Composite_outputs')
